In [ ]:
from pathlib import Path

import numpy as np
from numpy.typing import NDArray
from astropy.io import fits
from astropy.table import Table, vstack

from bloodmoon.mask import CodedMaskCamera

In [ ]:
DATASET: str =  'detected'

In [ ]:
def intersect_IDs(all_ids: NDArray, detected_ids: NDArray, uniqueIDs: bool = True) -> NDArray:
    """Extracts valid photon ID indexes within detected photons list."""
    _, _, idx_b = np.intersect1d(all_ids, detected_ids, assume_unique=uniqueIDs, return_indices=True)
    return idx_b


def merge_photons(
    phs_left: fits.FITS_rec,
    phs_right: fits.FITS_rec,
    save_to: str | Path | None = None,
    header: fits.Header | None = None,
) -> fits.FITS_rec:
    """Merges given data."""
    tl, tr = map(lambda x: Table(x), (phs_left, phs_right))
    merged_table = vstack([tl, tr])
    merged_hdu = fits.BinTableHDU(data=merged_table, header=header)
    print(f'Merged photon lists with total elements: {len(merged_table)}')

    if save_to is not None:
        print("# Saving merged photon list...")
        hdu_list = fits.HDUList([fits.PrimaryHDU(), merged_hdu])
        hdu_list.writeto(save_to, output_verify="fix+ignore")
        hdu_list.close()
        print("# Saving completed!")
    
    return merged_hdu.data